In [ ]:
!nvidia-smi

In [ ]:
!pip install ultralytics

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#from google.colab import files
#uploaded = files.upload()

In [ ]:
import zipfile
import os

# NEW PATH
zip_name = "/content/drive/MyDrive/plates_dataset.zip"
extract_path = "/content/new_dataset"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall(extract_path)

print(f"Extracted to {extract_path}")
print("Contents:", os.listdir(extract_path))

In [ ]:

import glob

yaml_files = glob.glob(f"{extract_path}/**/data.yaml", recursive=True)
print("Found data.yaml at:", yaml_files)

# Show the content so you can verify the classes
if yaml_files:
    with open(yaml_files[0], 'r') as f:
        print("\n--- data.yaml content ---")
        print(f.read())

In [ ]:
# 1. The correct base path where your images actually are
correct_path = "/content/new_dataset/plates_dataset"

# 2. Read the data.yaml, replace the path, and overwrite it
yaml_path = yaml_files[0]

with open(yaml_path, 'r') as f:
    lines = f.readlines()

with open(yaml_path, 'w') as f:
    for line in lines:
        if line.startswith('path:'):
            f.write(f'path: {correct_path}\n')  # Write the correct path
        else:
            f.write(line)

print("✅ Fixed data.yaml! New contents:")
print(open(yaml_path).read())

In [ ]:
#from ultralytics import YOLO

# 1. Load YOLO11 Nano
#model = YOLO("yolo11n.pt")

# 2. Train with anti-overfitting parameters
#model.train(
#    data=yaml_files[0],
#    epochs=100,
#    imgsz=1280,
#    batch=-1,
#    cache=False,
#    patience=20,

#    weight_decay=0.0005,
#    name="house_number_plate_yolo11"
#)

from ultralytics import YOLO
model = YOLO("yolo11n.pt")
model.train(
    data=yaml_files[0],
    epochs=100,
    imgsz=1280,
    batch=-1,
    cache=False,
    patience=20,
    weight_decay=0.0005,
    project="/content/drive/MyDrive/yolo_runs",
    name="yolo11n_1280_L4",
)

In [ ]:
from ultralytics import YOLO
model = YOLO("/content/drive/MyDrive/yolo_runs/yolo11n_1280_L4/weights/best.pt")
m = model.val(data=yaml_files[0], split="val", imgsz=1280, batch=8)
print(f"Precision {m.box.mp:.3f}  Recall {m.box.mr:.3f}  mAP50 {m.box.map50:.3f}  mAP50-95 {m.box.map:.3f}")

In [ ]:
m = model.val(data=yaml_files[0], split="test", imgsz=1280, batch=8)
print(f"Precision {m.box.mp:.3f}  Recall {m.box.mr:.3f}  mAP50 {m.box.map50:.3f}  mAP50-95 {m.box.map:.3f}")

In [ ]:
import shutil
shutil.copytree("/content/runs/detect/val", "/content/drive/MyDrive/yolo_runs/yolo11n_1280_L4_val_plots")
shutil.copytree("/content/runs/detect/val-2", "/content/drive/MyDrive/yolo_runs/yolo11n_1280_L4_test_plots")
print("Done")

In [ ]:
import os
from ultralytics import YOLO

WEIGHTS = "/content/runs/detect/house_number_plate_yolo11-2/weights/best.pt"

print("Exists:", os.path.exists(WEIGHTS))
print("Size (MB):", round(os.path.getsize(WEIGHTS)/1e6, 2))

model = YOLO(WEIGHTS)
print("Task:", model.task)
print("Classes:", model.names)
print("Model name (from checkpoint metadata):", getattr(model.model, 'yaml', {}).get('yaml_file', 'unknown'))

In [ ]:
from ultralytics import YOLO
m = YOLO("/content/runs/detect/house_number_plate_yolo11-2/weights/best.pt")

args = m.ckpt.get("train_args", {})
print("Model file:  ", args.get("model"))
print("Dataset:     ", args.get("data"))
print("Epochs:      ", args.get("epochs"))
print("Image size:  ", args.get("imgsz"))
print("Batch:       ", args.get("batch"))

In [ ]:
from ultralytics import YOLO
import os

# 1. Define paths
WEIGHTS = "/content/runs/detect/house_number_plate_yolo11-2/weights/best.pt"
DATASET_YAML = "/content/new_dataset/plates_dataset/data.yaml"

# 2. Check if files exist
if not os.path.exists(WEIGHTS):
    raise FileNotFoundError(f"Missing weights: {WEIGHTS}")
if not os.path.exists(DATASET_YAML):
    raise FileNotFoundError(f"Missing dataset yaml: {DATASET_YAML}")

# 3. Load model
print("Loading model...")
model = YOLO(WEIGHTS)

# 4. Run Validation
print("Running validation on the val split...")
metrics = model.val(
    data=DATASET_YAML,
    split="val",       # Force it to use the validation set
    imgsz=640,         # Same as training
    batch=16,
    conf=0.001,        # Low threshold to calculate full precision-recall curve (standard practice)
    iou=0.6,           # NMS IoU threshold
    save_json=False,   # Don't save COCO json unless you need it
    plots=True         # Generates confusion matrix, PR curve, etc.
)

# 5. Print the results clearly
print("\n" + "="*30)
print("VALIDATION RESULTS")
print("="*30)
print(f"Precision:  {metrics.box.mp:.3f}")
print(f"Recall:     {metrics.box.mr:.3f}")
print(f"mAP@50:     {metrics.box.map50:.3f}")
print(f"mAP@50-95:  {metrics.box.map:.3f}")

# 6. Where to find the plots
print(f"\nSaved plots (confusion matrix, curves, etc.) to:\n{metrics.save_dir}")

In [ ]:
import os
from IPython.display import Image, display

VAL_DIR = "/content/runs/detect/val"

print("Files in val folder:")
for f in sorted(os.listdir(VAL_DIR)):
    print(" ", f)

# Show the most important three
for name in ["confusion_matrix.png", "PR_curve.png", "val_batch0_pred.jpg"]:
    path = os.path.join(VAL_DIR, name)
    if os.path.exists(path):
        print(f"\n--- {name} ---")
        display(Image(path, width=800))
    else:
        print(f"\n{name} NOT FOUND")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from ultralytics import YOLO
import cv2
import os
import glob
from google.colab.patches import cv2_imshow

# 1. Paths
WEIGHTS = "/content/runs/detect/house_number_plate_yolo11-2/weights/best.pt"

# 2. Load model
model = YOLO(WEIGHTS)

# 3. Find images to test
# If you uploaded via Option 1, your images are in /content/ (e.g., /content/test.jpg)
# If you used Option 2 (Drive), change this path to your Drive folder (e.g., "/content/drive/MyDrive/MyImages/*.jpg")

# For Option 1 (Uploaded images), let's grab all images in the current directory:
image_paths = glob.glob("/content/*.jpg") + glob.glob("/content/*.png")

if len(image_paths) == 0:
    print("No images found! Please upload some images first.")
else:
    print(f"Found {len(image_paths)} image(s). Running inference...")

    # 4. Run predictions
    # conf=0.25 is the default. Lower it to 0.15 if you want to catch more plates (higher recall).
    results = model.predict(source=image_paths, conf=0.25, save=True)

    # 5. Display results
    for r in results:
        print(f"\n--- Image: {os.path.basename(r.path)} ---")
        print(f"Found {len(r.boxes)} plate(s).")

        for i, box in enumerate(r.boxes):
            conf = float(box.conf)
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            print(f"  Plate {i+1}: Confidence {conf:.2f} | Box: ({x1}, {y1}) to ({x2}, {y2})")

        # Plot and show the image with boxes
        annotated_img = r.plot()
        print("Visual Result:")
        cv2_imshow(annotated_img)

In [ ]:
import glob
images = glob.glob("/content/*.jpg") + glob.glob("/content/*.png")
for img in images:
    print("Found image at:", img)

In [ ]:
import shutil
import os

# 1. The location of your current trained weights
source_best = "/content/runs/detect/house_number_plate_yolo11-2/weights/best.pt"
source_last = "/content/runs/detect/house_number_plate_yolo11-2/weights/last.pt"

# 2. Your empty Google Drive folder
drive_folder = "/content/drive/MyDrive/house_number_plate_yolov11_weights"

# 3. Create the folder if it doesn't exist (just in case)
os.makedirs(drive_folder, exist_ok=True)

# 4. Copy the files
shutil.copy(source_best, os.path.join(drive_folder, "best.pt"))
shutil.copy(source_last, os.path.join(drive_folder, "last.pt"))

print("✅ Weights successfully copied to Google Drive!")
print(f"Location: {drive_folder}")

# 5. Verify they are there and check the size
for f in ["best.pt", "last.pt"]:
    path = os.path.join(drive_folder, f)
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1e6
        print(f"  Found {f} ({size_mb:.2f} MB)")